<a href="https://colab.research.google.com/github/vishaldeo71-art/bike-sharing-demand-ml/blob/main/bike_sharing_ml1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================
# IMPORTING REQUIRED LIBRARIES
# ==============================

# Pandas is used for loading and manipulating datasets
import pandas as pd

# NumPy is used for numerical calculations
import numpy as np

# Matplotlib is used for creating graphs
import matplotlib.pyplot as plt

# Seaborn is used for better-looking statistical visualizations
import seaborn as sns

# Used to split data into training and testing sets
from sklearn.model_selection import train_test_split

# Used to scale features
from sklearn.preprocessing import StandardScaler

# Machine Learning Models
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression

from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import plot_tree

from sklearn.ensemble import RandomForestRegressor

from sklearn.neighbors import KNeighborsRegressor

# Regression Evaluation Metrics
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

# Classification Evaluation Metrics
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

# Used for saving models if required
import joblib

print("All libraries imported successfully!")

In [ ]:
# =====================================
# UPLOAD THE DATASET IN GOOGLE COLAB
# =====================================

from google.colab import files

uploaded = files.upload()

In [ ]:
# ==============================
# LOADING THE DATASET
# ==============================

# Read the CSV file
df = pd.read_csv("train.csv")

print("Dataset loaded successfully!")

In [ ]:
# First 5 rows
print("FIRST 5 ROWS")
display(df.head())

# Last 5 rows
print("LAST 5 ROWS")
display(df.tail())

In [ ]:
# ==============================
# DATASET INSPECTION
# ==============================

# Dataset shape
print("Dataset Shape:")
print(df.shape)

# Column names
print("\nColumn Names:")
print(df.columns.tolist())

# Data types
print("\nData Types:")
print(df.dtypes)

# Dataset information
print("\nDataset Information:")
df.info()

# Basic statistics
print("\nBasic Statistics:")
display(df.describe())

In [ ]:
# ==============================
# IDENTIFYING COLUMN TYPES
# ==============================

numerical_columns = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

categorical_columns = df.select_dtypes(include=["object"]).columns.tolist()

print("Numerical Columns:")
print(numerical_columns)

print("\nCategorical Columns:")
print(categorical_columns)

In [ ]:
# ==============================
# UNIQUE VALUES IN IMPORTANT COLUMNS
# ==============================

important_columns = ["season", "weather", "holiday", "workingday"]

for column in important_columns:
    if column in df.columns:
        print(f"\nColumn: {column}")
        print("Number of unique values:", df[column].nunique())
        print("Unique values:", sorted(df[column].unique()))

In [ ]:
# ==============================
# INITIAL DATASET OBSERVATIONS
# ==============================

print("Total number of rows:", df.shape[0])
print("Total number of columns:", df.shape[1])

print("\nBike rental count statistics:")
print(df["count"].describe())

print("\nSeason distribution:")
print(df["season"].value_counts().sort_index())

print("\nWeather distribution:")
print(df["weather"].value_counts().sort_index())

print("\nWorking day distribution:")
print(df["workingday"].value_counts().sort_index())

In [ ]:
# ==============================
# CHECKING MISSING VALUES
# ==============================

missing_values = df.isnull().sum()

print("Missing values in each column:")
print(missing_values)

total_missing = missing_values.sum()

print("\nTotal Missing Values:", total_missing)

if total_missing == 0:
    print("No missing values found in the dataset.")
else:
    print("Missing values found. They need to be handled.")

In [ ]:
# ==============================
# HANDLING MISSING VALUES
# ==============================

for column in df.columns:

    if df[column].isnull().sum() > 0:

        # Numerical columns
        if df[column].dtype in ["int64", "float64"]:

            df[column] = df[column].fillna(df[column].median())

        # Categorical columns
        else:

            df[column] = df[column].fillna(df[column].mode()[0])

print("Missing value handling completed.")

print("\nRemaining missing values:")
print(df.isnull().sum())

In [ ]:
# ==============================
# CHECKING DUPLICATE ROWS
# ==============================

duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

if duplicate_count > 0:

    df = df.drop_duplicates()

    print("Duplicate rows removed.")
    print("New dataset shape:", df.shape)

else:

    print("No duplicate rows found. No removal required.")

In [ ]:
# ==============================
# CHECKING INVALID VALUES
# ==============================

# Check negative values in columns where negatives are not expected

columns_to_check = [
    "temp",
    "atemp",
    "humidity",
    "windspeed",
    "count"
]

for column in columns_to_check:

    if column in df.columns:

        negative_values = (df[column] < 0).sum()

        print(f"{column}: Negative values = {negative_values}")


In [ ]:
# ==============================
# CHECKING INVALID VALUES
# ==============================

# Check negative values in columns where negatives are not expected

columns_to_check = [
    "temp",
    "atemp",
    "humidity",
    "windspeed",
    "count"
]

for column in columns_to_check:

    if column in df.columns:

        negative_values = (df[column] < 0).sum()

        print(f"{column}: Negative values = {negative_values}")

In [ ]:
# ==============================
# DATETIME FEATURE ENGINEERING
# ==============================

# Convert datetime column into datetime format
df["datetime"] = pd.to_datetime(df["datetime"])

# Extract useful features

df["year"] = df["datetime"].dt.year

df["month"] = df["datetime"].dt.month

df["day"] = df["datetime"].dt.day

df["hour"] = df["datetime"].dt.hour

df["day_of_week"] = df["datetime"].dt.dayofweek

print("Datetime features created successfully!")

display(
    df[
        [
            "datetime",
            "year",
            "month",
            "day",
            "hour",
            "day_of_week"
        ]
    ].head()
)

In [ ]:
# ==============================
# VISUALIZATION 1
# DISTRIBUTION OF BIKE DEMAND
# ==============================

plt.figure(figsize=(10, 6))

sns.histplot(df["count"], bins=40, kde=True)

plt.title("Distribution of Bike Rental Demand")

plt.xlabel("Bike Rental Count")

plt.ylabel("Frequency")

plt.show()

In [ ]:
# ==============================
# VISUALIZATION 2
# TEMPERATURE VS BIKE DEMAND
# ==============================

plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="temp",
    y="count",
    alpha=0.5
)

plt.title("Relationship Between Temperature and Bike Rental Demand")

plt.xlabel("Temperature")

plt.ylabel("Bike Rental Count")

plt.show()

In [ ]:
# ==============================
# VISUALIZATION 3
# AVERAGE BIKE DEMAND BY SEASON
# ==============================

season_average = (
    df.groupby("season")["count"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(8, 5))

sns.barplot(
    data=season_average,
    x="season",
    y="count"
)

plt.title("Average Bike Rental Demand by Season")

plt.xlabel("Season")

plt.ylabel("Average Bike Rental Count")

plt.show()


In [ ]:
# ==============================
# CORRELATION HEATMAP
# ==============================

plt.figure(figsize=(12, 8))

numeric_df = df.select_dtypes(include=np.number)

sns.heatmap(
    numeric_df.corr(),
    cmap="coolwarm",
    annot=False
)

plt.title("Correlation Heatmap of Numerical Features")

plt.show()

In [ ]:
# ==============================
# CREATING FEATURES AND TARGET
# ==============================

# Columns that must NOT be used as input features

columns_to_remove = [
    "datetime",      # Raw datetime removed
    "count",         # This is the target
    "casual",        # Prevent data leakage
    "registered"     # Prevent data leakage
]

# Create feature dataset

X_reg = df.drop(
    columns=columns_to_remove,
    errors="ignore"
)

# Regression target

y_reg = df["count"]

print("Features used for Regression:")

print(X_reg.columns.tolist())

print("\nFeature Dataset Shape:", X_reg.shape)

print("Target Shape:", y_reg.shape)

In [ ]:
# ==============================
# DATA LEAKAGE CHECK
# ==============================

forbidden_columns = [
    "casual",
    "registered",
    "count",
    "datetime"
]

print("Checking for forbidden columns...")

for column in forbidden_columns:

    if column in X_reg.columns:

        print(f"WARNING: {column} is present!")

    else:

        print(f"PASS: {column} is NOT present.")

In [ ]:
# ==============================
# TRAIN-TEST SPLIT
# ==============================

X_train, X_test, y_train, y_test = train_test_split(
    X_reg,
    y_reg,
    test_size=0.20,
    random_state=42
)

print("Training Feature Shape:", X_train.shape)

print("Testing Feature Shape:", X_test.shape)

print("Training Target Shape:", y_train.shape)

print("Testing Target Shape:", y_test.shape)

In [ ]:
# ==============================
# REGRESSION EVALUATION FUNCTION
# ==============================

def evaluate_regression_model(
    model_name,
    y_actual,
    y_predicted
):

    mae = mean_absolute_error(
        y_actual,
        y_predicted
    )

    mse = mean_squared_error(
        y_actual,
        y_predicted
    )

    rmse = np.sqrt(mse)

    r2 = r2_score(
        y_actual,
        y_predicted
    )

    print(f"\n----- {model_name} -----")

    print("MAE:", round(mae, 2))

    print("MSE:", round(mse, 2))

    print("RMSE:", round(rmse, 2))

    print("R² Score:", round(r2, 4))

    return {
        "Model": model_name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    }

In [ ]:
# ==============================
# MODEL A
# LINEAR REGRESSION
# ==============================

# Create model

linear_model = LinearRegression()

# Train model

linear_model.fit(
    X_train,
    y_train
)

# Make predictions

linear_predictions = linear_model.predict(
    X_test
)

# Evaluate model

linear_results = evaluate_regression_model(
    "Linear Regression",
    y_test,
    linear_predictions
)

In [ ]:
# ==============================
# ACTUAL VS PREDICTED
# LINEAR REGRESSION
# ==============================

plt.figure(figsize=(8, 6))

plt.scatter(
    y_test,
    linear_predictions,
    alpha=0.5
)

plt.xlabel("Actual Bike Rental Count")

plt.ylabel("Predicted Bike Rental Count")

plt.title("Linear Regression: Actual vs Predicted")

# Reference line

minimum = min(
    y_test.min(),
    linear_predictions.min()
)

maximum = max(
    y_test.max(),
    linear_predictions.max()
)

plt.plot(
    [minimum, maximum],
    [minimum, maximum],
    "r--"
)

plt.show()

In [ ]:
# ==============================
# CREATE CLASSIFICATION TARGET
# ==============================

# Find median bike rental count

median_count = df["count"].median()

print("Median Bike Rental Count:", median_count)

# Create binary classification target

df["high_demand"] = (
    df["count"] >= median_count
).astype(int)

print("\nHigh Demand Class Distribution:")

print(
    df["high_demand"]
    .value_counts()
)

In [ ]:
# ==============================
# FEATURES FOR LOGISTIC REGRESSION
# ==============================

# Use the same safe features

X_classification = X_reg.copy()

# Classification target

y_classification = df["high_demand"]

# Train-test split

X_train_class, X_test_class, y_train_class, y_test_class = train_test_split(
    X_classification,
    y_classification,
    test_size=0.20,
    random_state=42,
    stratify=y_classification
)

print("Classification Train Shape:", X_train_class.shape)

print("Classification Test Shape:", X_test_class.shape)

In [ ]:
# ==============================
# FEATURE SCALING
# LOGISTIC REGRESSION
# ==============================

scaler_logistic = StandardScaler()

# Fit ONLY on training data

X_train_class_scaled = scaler_logistic.fit_transform(
    X_train_class
)

# Transform testing data

X_test_class_scaled = scaler_logistic.transform(
    X_test_class
)

print("Scaling completed successfully!")

In [ ]:
# ==============================
# MODEL B
# LOGISTIC REGRESSION
# ==============================

logistic_model = LogisticRegression(
    max_iter=1000
)

# Train model

logistic_model.fit(
    X_train_class_scaled,
    y_train_class
)

# Predictions

logistic_predictions = logistic_model.predict(
    X_test_class_scaled
)

# Accuracy

logistic_accuracy = accuracy_score(
    y_test_class,
    logistic_predictions
)

print(
    "Logistic Regression Accuracy:",
    round(logistic_accuracy, 4)
)

In [ ]:
# ==============================
# CONFUSION MATRIX
# ==============================

confusion = confusion_matrix(
    y_test_class,
    logistic_predictions
)

display_matrix = ConfusionMatrixDisplay(
    confusion_matrix=confusion,
    display_labels=["Low Demand", "High Demand"]
)

display_matrix.plot()

plt.title("Logistic Regression Confusion Matrix")

plt.show()

In [ ]:
# ==============================
# MODEL C
# DECISION TREE REGRESSOR
# ==============================

decision_tree_model = DecisionTreeRegressor(
    max_depth=8,
    random_state=42
)

# Train

decision_tree_model.fit(
    X_train,
    y_train
)

# Predict

decision_tree_predictions = decision_tree_model.predict(
    X_test
)

# Evaluate

decision_tree_results = evaluate_regression_model(
    "Decision Tree",
    y_test,
    decision_tree_predictions
)

In [ ]:
# ==============================
# DECISION TREE VISUALIZATION
# ==============================

plt.figure(figsize=(20, 10))

plot_tree(
    decision_tree_model,
    feature_names=X_train.columns,
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=8
)

plt.title(
    "Decision Tree Visualization (Simplified to Depth 3)"
)

plt.show()

In [ ]:
# ==============================
# MODEL D
# RANDOM FOREST REGRESSOR
# ==============================

random_forest_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Train

random_forest_model.fit(
    X_train,
    y_train
)

# Predictions

random_forest_predictions = random_forest_model.predict(
    X_test
)

# Evaluation

random_forest_results = evaluate_regression_model(
    "Random Forest",
    y_test,
    random_forest_predictions
)


In [ ]:
# ==============================
# FEATURE IMPORTANCE
# ==============================

feature_importance = pd.DataFrame({

    "Feature": X_train.columns,

    "Importance": random_forest_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print("Top 3 Important Features:")

display(
    feature_importance.head(3)
)

In [ ]:
# ==============================
# FEATURE IMPORTANCE GRAPH
# ==============================

plt.figure(figsize=(10, 6))

sns.barplot(
    data=feature_importance,
    x="Importance",
    y="Feature"
)

plt.title(
    "Random Forest Feature Importance"
)

plt.show()

In [ ]:
# ==============================
# FEATURE SCALING FOR KNN
# ==============================

scaler_knn = StandardScaler()

# Fit scaler only on training data

X_train_scaled = scaler_knn.fit_transform(
    X_train
)

# Transform testing data

X_test_scaled = scaler_knn.transform(
    X_test
)

print("KNN feature scaling completed!")

In [ ]:
# ==============================
# MODEL E
# K-NEAREST NEIGHBORS
# ==============================

# K values to test

k_values = [3, 5, 7]

knn_results_list = []

for k in k_values:

    print(f"\nTesting K = {k}")

    # Create model

    knn_model = KNeighborsRegressor(
        n_neighbors=k
    )

    # Train

    knn_model.fit(
        X_train_scaled,
        y_train
    )

    # Predict

    knn_predictions = knn_model.predict(
        X_test_scaled
    )

    # Evaluate

    result = evaluate_regression_model(
        f"KNN (K={k})",
        y_test,
        knn_predictions
    )

    # Store K value

    result["K"] = k

    knn_results_list.append(
        result
    )

In [ ]:
# ==============================
# KNN COMPARISON
# ==============================

knn_results_df = pd.DataFrame(
    knn_results_list
)

display(
    knn_results_df[
        [
            "K",
            "MAE",
            "RMSE",
            "R2"
        ]
    ]
)

In [ ]:
# ==============================
# BEST K VALUE
# ==============================

best_knn = knn_results_df.loc[
    knn_results_df["R2"].idxmax()
]

print("Best K Value:", int(best_knn["K"]))

print(
    "Best KNN R² Score:",
    round(best_knn["R2"], 4)
)

In [ ]:
# ==============================
# FINAL KNN MODEL
# ==============================

best_k = int(best_knn["K"])

knn_final_model = KNeighborsRegressor(
    n_neighbors=best_k
)

knn_final_model.fit(
    X_train_scaled,
    y_train
)

knn_final_predictions = knn_final_model.predict(
    X_test_scaled
)

knn_final_results = evaluate_regression_model(
    f"KNN (Best K={best_k})",
    y_test,
    knn_final_predictions
)

In [ ]:
# ==============================
# FINAL MODEL COMPARISON
# ==============================

comparison_results = pd.DataFrame([

    {
        "Model": "Linear Regression",
        "Problem Type": "Regression",
        "MAE": linear_results["MAE"],
        "RMSE": linear_results["RMSE"],
        "R2": linear_results["R2"],
        "Accuracy": "N/A"
    },

    {
        "Model": "Logistic Regression",
        "Problem Type": "Classification",
        "MAE": "N/A",
        "RMSE": "N/A",
        "R2": "N/A",
        "Accuracy": logistic_accuracy
    },

    {
        "Model": "Decision Tree",
        "Problem Type": "Regression",
        "MAE": decision_tree_results["MAE"],
        "RMSE": decision_tree_results["RMSE"],
        "R2": decision_tree_results["R2"],
        "Accuracy": "N/A"
    },

    {
        "Model": "Random Forest",
        "Problem Type": "Regression",
        "MAE": random_forest_results["MAE"],
        "RMSE": random_forest_results["RMSE"],
        "R2": random_forest_results["R2"],
        "Accuracy": "N/A"
    },

    {
        "Model": f"KNN (K={best_k})",
        "Problem Type": "Regression",
        "MAE": knn_final_results["MAE"],
        "RMSE": knn_final_results["RMSE"],
        "R2": knn_final_results["R2"],
        "Accuracy": "N/A"
    }
])

display(comparison_results)

In [ ]:
# ==============================
# BEST AND WORST REGRESSION MODEL
# ==============================

regression_models = comparison_results[
    comparison_results["Problem Type"] == "Regression"
].copy()

# Convert R2 to numeric

regression_models["R2"] = pd.to_numeric(
    regression_models["R2"]
)

best_model = regression_models.loc[
    regression_models["R2"].idxmax()
]

worst_model = regression_models.loc[
    regression_models["R2"].idxmin()
]

print("BEST REGRESSION MODEL")

print(best_model)

print("\nWORST REGRESSION MODEL")

print(worst_model)

In [ ]:
# ==============================
# SELECT BEST MODEL PREDICTIONS
# ==============================

prediction_dictionary = {

    "Linear Regression": linear_predictions,

    "Decision Tree": decision_tree_predictions,

    "Random Forest": random_forest_predictions,

    f"KNN (K={best_k})": knn_final_predictions
}

best_model_name = best_model["Model"]

best_predictions = prediction_dictionary[
    best_model_name
]

print(
    "Performing Error Analysis for:",
    best_model_name
)

In [ ]:
# ==============================
# ERROR ANALYSIS TABLE
# ==============================

error_analysis = pd.DataFrame({

    "Actual Count": y_test.values,

    "Predicted Count": best_predictions
})

# Calculate absolute error

error_analysis["Absolute Error"] = abs(
    error_analysis["Actual Count"]
    -
    error_analysis["Predicted Count"]
)

# Show sample predictions

display(
    error_analysis.head(10)
)

In [ ]:
# ==============================
# ERROR STATISTICS
# ==============================

absolute_errors = error_analysis[
    "Absolute Error"
]

print(
    "Mean Absolute Error:",
    round(absolute_errors.mean(), 2)
)

print(
    "Minimum Absolute Error:",
    round(absolute_errors.min(), 2)
)

print(
    "Maximum Absolute Error:",
    round(absolute_errors.max(), 2)
)

print(
    "Median Absolute Error:",
    round(absolute_errors.median(), 2)
)

In [ ]:
# ==============================
# ERROR RANGE ANALYSIS
# ==============================

total_predictions = len(
    absolute_errors
)

within_10 = (
    (absolute_errors <= 10).sum()
    /
    total_predictions
    *
    100
)

within_25 = (
    (absolute_errors <= 25).sum()
    /
    total_predictions
    *
    100
)

within_50 = (
    (absolute_errors <= 50).sum()
    /
    total_predictions
    *
    100
)

more_than_50 = (
    (absolute_errors > 50).sum()
    /
    total_predictions
    *
    100
)

error_ranges = pd.DataFrame({

    "Error Range": [
        "Within ±10",
        "Within ±25",
        "Within ±50",
        "More than ±50"
    ],

    "Percentage": [
        within_10,
        within_25,
        within_50,
        more_than_50
    ]
})

display(error_ranges)

In [ ]:
# ==============================
# NON-OVERLAPPING ERROR RANGES
# ==============================

range_0_10 = (
    (absolute_errors <= 10).sum()
    /
    total_predictions
    *
    100
)

range_11_25 = (
    ((absolute_errors > 10) & (absolute_errors <= 25)).sum()
    /
    total_predictions
    *
    100
)

range_26_50 = (
    ((absolute_errors > 25) & (absolute_errors <= 50)).sum()
    /
    total_predictions
    *
    100
)

range_above_50 = (
    (absolute_errors > 50).sum()
    /
    total_predictions
    *
    100
)

error_distribution = pd.DataFrame({

    "Error Range": [
        "0-10",
        "11-25",
        "26-50",
        ">50"
    ],

    "Percentage": [
        range_0_10,
        range_11_25,
        range_26_50,
        range_above_50
    ]
})

display(error_distribution)

In [ ]:
# ==============================
# ERROR DISTRIBUTION GRAPH
# ==============================

plt.figure(figsize=(10, 6))

sns.histplot(
    absolute_errors,
    bins=30,
    kde=True
)

plt.title(
    f"Prediction Error Distribution - {best_model_name}"
)

plt.xlabel(
    "Absolute Prediction Error"
)

plt.ylabel(
    "Frequency"
)

plt.show()

In [ ]:
# ==============================
# SAVE MODEL RESULTS
# ==============================

comparison_results.to_csv(
    "model_results_summary.csv",
    index=False
)

print(
    "Model results saved successfully!"
)

In [ ]:
# ==============================
# FINAL R² SCORE SUMMARY
# ==============================

print("\n========== FINAL R² SCORES ==========\n")

for _, row in regression_models.iterrows():

    print(
        f"{row['Model']} "
        f"--> R² Score: "
        f"{row['R2']:.4f}"
    )

print("\n====================================")

print(
    "\nBEST REGRESSION MODEL:"
)

print(
    best_model["Model"]
)

print(
    "\nBEST R² SCORE:"
)

print(
    round(
        best_model["R2"],
        4
    )
)